In [11]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [12]:
df=pd.read_csv('course_lead_scoring_2026.csv')
df.dtypes


lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [13]:
df.isnull().sum()
categorical=df.select_dtypes(include=['str']).columns.tolist()
numerical=df.select_dtypes(exclude=['str']).columns.tolist()
df[categorical]=df[categorical].fillna('NA')
df[numerical]=df[numerical].fillna(0.0)

In [14]:
# Question 1
# What is the most frequent observation (mode) for the column industry?

# NA
# technology
# healthcare
# retail

df.industry.mode()

0    technology
Name: industry, dtype: str

In [15]:
# Question 2 
# Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.

# What are the two features that have the biggest correlation?

# interaction_count and lead_score
# number_of_courses_viewed and lead_score
# number_of_courses_viewed and interaction_count
# annual_income and interaction_count
# Only consider the pairs above when answering this question.

corr_matrix=df[numerical].corr()
print(corr_matrix.loc['interaction_count', 'lead_score'])
print(corr_matrix.loc['number_of_courses_viewed', 'lead_score'])
print(corr_matrix.loc['number_of_courses_viewed', 'interaction_count'])
print(corr_matrix.loc['annual_income', 'interaction_count'])


0.9157457980418697
0.7572042392104886
0.7216090038937035
0.12284235135955959


In [16]:
df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=42
)

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

del df_train['converted']
del df_val['converted']
del df_test['converted']

In [17]:
# Question 3
# Calculate the mutual information score between converted and other categorical variables in the dataset. Use the training set only.
# Round the scores to 2 decimals using round(score, 2).
# Which of these variables has the biggest mutual information score?

# industry
# location
# lead_source
# employment_status

categorical_mi = [
    'industry',
    'location',
    'lead_source',
    'employment_status'
]

def calculate_mi(series):
    return round(mutual_info_score(series, y_train), 2)

mi_scores = df_train[categorical_mi].apply(calculate_mi)

print(mi_scores.sort_values(ascending=False))

lead_source          0.03
employment_status    0.02
location             0.00
industry             0.00
dtype: float64


In [18]:
# Question 4
# Now let's train a logistic regression.
# Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
# Fit the model on the training dataset.
# To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
# model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
# Calculate the accuracy on the validation dataset and round it to 2 decimal digits.
# What accuracy did you get?

# 0.55
# 0.65
# 0.75
# 0.85

dv = DictVectorizer(sparse=False)

train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

model = LogisticRegression(
    solver='liblinear',
    C=1.0,
    max_iter=1000,
    random_state=42
)

model.fit(X_train, y_train)

y_pred = model.predict(X_val)

accuracy = round(accuracy_score(y_val, y_pred), 2)
print(accuracy)

0.65


In [19]:
# Question 5
# Let's find the least useful feature using the feature elimination technique.
# Train a model using the same features and parameters as in Q4 (without rounding).
# Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
# For each feature, calculate the difference between the original accuracy and the accuracy without the feature.
# Which of following feature has the smallest difference?

# 'lead_source'
# 'number_of_courses_viewed'
# 'interaction_count'
# Note: The difference doesn't have to be positive.

original_accuracy = accuracy_score(y_val, y_pred)
print("Original accuracy:", original_accuracy)

# Eliminar una característica cada vez
features = [
    'lead_source',
    'number_of_courses_viewed',
    'interaction_count'
]

for feature in features:
    train_dicts_reduced = df_train.drop(columns=[feature]).to_dict(orient='records')
    val_dicts_reduced = df_val.drop(columns=[feature]).to_dict(orient='records')

    dv_reduced = DictVectorizer(sparse=False)
    X_train_reduced = dv_reduced.fit_transform(train_dicts_reduced)
    X_val_reduced = dv_reduced.transform(val_dicts_reduced)

    model_reduced = LogisticRegression(
        solver='liblinear',
        C=1.0,
        max_iter=1000,
        random_state=42
    )

    model_reduced.fit(X_train_reduced, y_train)
    y_pred_reduced = model_reduced.predict(X_val_reduced)

    accuracy_reduced = accuracy_score(y_val, y_pred_reduced)
    difference = original_accuracy - accuracy_reduced

    print(
        feature,
        "accuracy:", accuracy_reduced,
        "difference:", difference
    )

Original accuracy: 0.645
lead_source accuracy: 0.642 difference: 0.0030000000000000027
number_of_courses_viewed accuracy: 0.643 difference: 0.0020000000000000018
interaction_count accuracy: 0.601 difference: 0.04400000000000004


In [ ]:
# Question 6
# Now let's train a regularized logistic regression.
# Let's try the following values of the parameter C: [0.000001, 0.00001, 0.0001, 0.001].
# Train models using all the features as in Q4.
# Calculate the accuracy on the validation dataset and round it to 3 decimal digits.
# Which of these C leads to the best accuracy on the validation set?

# 0.000001
# 0.00001
# 0.0001
# 0.001
# Note: If there are multiple options, select the smallest C.

# Preparar los datos como en Q4
dv = DictVectorizer(sparse=False)

train_dicts = df_train.to_dict(orient='records')
val_dicts = df_val.to_dict(orient='records')

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

# Probar los valores de C
Cs = [0.000001, 0.00001, 0.0001, 0.001]

results = {}

for C in Cs:
    model = LogisticRegression(
        solver='liblinear',
        C=C,
        max_iter=1000,
        random_state=42
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_val)

    accuracy = round(accuracy_score(y_val, y_pred), 3)
    results[C] = accuracy

    print(f'C={C}: accuracy={accuracy}')

best_C = min(results, key=lambda c: (-results[c], c))
print(best_C) 

C=1e-06: accuracy=0.598
C=1e-05: accuracy=0.598
C=0.0001: accuracy=0.613
C=0.001: accuracy=0.645
0.001
